# Notebook 06 — Optuna Bayesian Optimization & Walk-Forward Validation

Optimizes 5 strategy parameters on the TRAIN period, validates on VAL, reports on TEST.
Walk-forward uses 5 rolling windows (50 trials each) to assess robustness.

## 6.1 — Setup

In [1]:
import sys, os, time, warnings
import subprocess
# Install optuna if needed
try:
    import optuna
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'optuna', '--quiet'])
    import optuna

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Resolve project root (works from notebooks/ or project root)
_here = os.getcwd()
PROJECT_ROOT = _here if os.path.isdir(os.path.join(_here, 'data')) else os.path.dirname(_here)
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

from backtest import run_backtest, calc_metrics
from report_utils import set_plot_style
set_plot_style()

DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed') + '/'
FIG_DIR  = os.path.join(PROJECT_ROOT, 'reports', 'figures') + '/'
os.makedirs(FIG_DIR, exist_ok=True)

TRAIN_START = '2018-01-01'; TRAIN_END = '2020-12-31'
VAL_START   = '2021-01-01'; VAL_END   = '2021-12-31'
TEST_START  = '2022-01-01'; TEST_END  = '2024-12-31'

print(f'optuna {optuna.__version__}  |  Project root: {PROJECT_ROOT}')


/Users/aaravaguru/Library/CloudStorage/OneDrive-IndianaUniversity/Quant Project 1 - DriftFIre/venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


optuna 4.8.0  |  Project root: /Users/aaravaguru/Library/CloudStorage/OneDrive-IndianaUniversity/Quant Project 1 - DriftFIre


## 6.2 — Load Data & Pre-compute Shift Arrays

In [2]:
df = pd.read_parquet(DATA_DIR + 'features.parquet')
df['date'] = pd.to_datetime(df['date'])
df['ticker'] = df['ticker'].astype('category')
print(f'Loaded: {len(df):,} rows x {df.shape[1]} cols')

# Sort once — all subsequent operations rely on this order
df_sorted = df.sort_values(['ticker', 'date']).reset_index(drop=True)

print('Pre-computing per-ticker shift arrays (lags 1-7)...')
t0 = time.time()

# Pre-compute grouped shifts of vol_ratio (lags 1..7)
grp = df_sorted.groupby('ticker', sort=False)
_vr_shifts = {}
for lag in range(1, 8):
    _vr_shifts[lag] = grp['vol_ratio'].shift(lag).values  # numpy array

# Pre-compute shift(1) of pct_from_ema8 (for tight consolidation)
_pem8_s1 = grp['pct_from_ema8'].shift(1).values

# Static boolean arrays (pre-computed, immutable per trial)
_ema_ok      = df_sorted['ema_aligned'].values.astype(bool)
_screener_ok = df_sorted['screener_pass'].values.astype(bool)
_sector_ok   = df_sorted['sector_top3'].values.astype(bool)
_is_stock    = (~df_sorted['is_etf'].values.astype(bool))
_regime_score = df_sorted['regime_score'].values   # already lagged in features.parquet
_vol_ratio   = df_sorted['vol_ratio'].values       # current day (for spike detection)

print(f'Pre-computation done in {time.time()-t0:.2f}s')
print(f'Arrays shape: {_vol_ratio.shape}')


Loaded: 823,610 rows x 49 cols
Pre-computing per-ticker shift arrays (lags 1-7)...
Pre-computation done in 0.04s
Arrays shape: (823610,)


## 6.3 — Parametric Signal Builder

In [3]:
def build_signal(vol_window: int, ema_prox: float,
                 spike_mult: float, regime_thr: int) -> np.ndarray:
    """Build a boolean signal array from pre-computed shift arrays.
    
    Uses the pre-computed shifted arrays (_vr_shifts, _pem8_s1) to avoid
    per-trial groupby overhead.  ~0.003s per call for 823k-row dataset.
    
    Parameters
    ----------
    vol_window   : int   Number of prior days volume must be below avg (2-7)
    ema_prox     : float Max |pct_from_ema8| for tight consolidation (0.005-0.04)
    spike_mult   : float Today's vol_ratio threshold for breakout (1.2-3.0)
    regime_thr   : int   Minimum regime_score for trading (1-4)
    
    Returns
    -------
    np.ndarray of bool, same length as df_sorted
    """
    # Vol dryup: vol_window consecutive prior days all had vol < 20d avg
    vol_dryup = np.ones(len(df_sorted), dtype=bool)
    for lag in range(1, int(vol_window) + 1):
        arr = _vr_shifts[lag]
        # NaN in any lag → can't confirm dryup → False
        vol_dryup &= (~np.isnan(arr)) & (arr < 1.0)
    
    # Tight consolidation: yesterday's |pct_from_ema8| <= ema_prox
    tight = np.abs(_pem8_s1) <= ema_prox
    tight[np.isnan(_pem8_s1)] = False
    
    # Vol spike: today's vol_ratio >= spike_mult
    spike = _vol_ratio >= spike_mult
    spike[np.isnan(_vol_ratio)] = False
    
    # Regime filter (already lagged in features.parquet)
    regime_ok = _regime_score >= regime_thr
    
    signal = (_is_stock & _screener_ok & _ema_ok &
              vol_dryup & tight & spike & regime_ok & _sector_ok)
    return signal

# Quick sanity check
test_sig = build_signal(3, 0.02, 1.5, 2)
print(f'Sanity check (baseline params): {test_sig.sum()} signals  '
      f'(expect ~109 matching original entry_signal)')
print(f'build_signal timing: ', end='')
t0 = time.time()
for _ in range(10):
    build_signal(3, 0.02, 1.5, 2)
print(f'{(time.time()-t0)/10*1000:.1f} ms per call')


Sanity check (baseline params): 102 signals  (expect ~109 matching original entry_signal)
build_signal timing: 3.8 ms per call


## 6.4 — Optuna Objective Function

In [4]:
# Pre-filter df to TRAIN period for faster backtest per trial
_train_mask = (
    (df_sorted['date'] >= pd.Timestamp(TRAIN_START)) &
    (df_sorted['date'] <= pd.Timestamp(TRAIN_END))
)
df_train_base = df_sorted[_train_mask].copy().reset_index(drop=True)
train_idx = np.where(_train_mask.values)[0]  # indices into df_sorted for train rows

print(f'Train period rows: {len(df_train_base):,}')

def make_objective(df_base, train_indices, period_start, period_end, n_min_trades=15):
    """Factory returning an Optuna objective for a given period."""
    
    def objective(trial):
        vol_window = trial.suggest_int('vol_consol_days', 2, 7)
        ema_prox   = trial.suggest_float('ema_proximity', 0.005, 0.04)
        spike_mult = trial.suggest_float('vol_spike_mult', 1.2, 3.0)
        regime_thr = trial.suggest_int('regime_threshold', 1, 4)
        max_hold   = trial.suggest_int('max_hold_days', 5, 30)
        
        # Compute signal on full df_sorted (for proper shift boundary handling)
        signal_full = build_signal(vol_window, ema_prox, spike_mult, regime_thr)
        
        # Extract signal values for the target period
        signal_period = signal_full[train_indices]
        
        if signal_period.sum() < n_min_trades:
            return -1.0  # Penalty: too few trades
        
        # Inject into base df
        df_run = df_base.copy()
        df_run['_opt_signal'] = signal_period
        
        # Run backtest (no start/end date needed — df already filtered)
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            trades, equity = run_backtest(df_run, signal_col='_opt_signal',
                                          max_hold_days=max_hold)
        
        if trades is None or len(trades) < n_min_trades:
            return -1.0
        
        m = calc_metrics(trades, equity, label='train', signal='opt')
        sharpe = m.get('sharpe', -1.0)
        if np.isnan(sharpe):
            return -1.0
        return float(sharpe)
    
    return objective

train_objective = make_objective(df_train_base, train_idx,
                                 TRAIN_START, TRAIN_END, n_min_trades=15)
print('Objective function ready.')


Train period rows: 350,987
Objective function ready.


## 6.5 — Run Main Optimization (150 trials, Train 2018-2020)

In [5]:
N_TRIALS = 150
print(f'Running {N_TRIALS} Optuna trials on TRAIN period (2018-2020)...')
t0 = time.time()

study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),
    study_name='driftfire_main',
)
study.optimize(train_objective, n_trials=N_TRIALS, show_progress_bar=False)

elapsed = time.time() - t0
print(f'Done in {elapsed:.1f}s ({elapsed/N_TRIALS:.2f}s/trial)')
print()
print('Best trial:')
best = study.best_trial
print(f'  Sharpe (train): {best.value:.3f}')
print('  Parameters:')
for k, v in best.params.items():
    print(f'    {k}: {v}')


Running 150 Optuna trials on TRAIN period (2018-2020)...


Done in 32.7s (0.22s/trial)

Best trial:
  Sharpe (train): 1.010
  Parameters:
    vol_consol_days: 3
    ema_proximity: 0.028401567185994646
    vol_spike_mult: 1.2048452327966348
    regime_threshold: 1
    max_hold_days: 12


## 6.6 — Evaluate Best Parameters on All Periods

In [6]:
best_params = study.best_trial.params
print('Best parameters:', best_params)
print()

# Build signal with best params
best_signal = build_signal(
    vol_window = best_params['vol_consol_days'],
    ema_prox   = best_params['ema_proximity'],
    spike_mult = best_params['vol_spike_mult'],
    regime_thr = best_params['regime_threshold'],
)
df_sorted['entry_optuna'] = best_signal

# Signal counts by period
for period, start, end in [
    ('train', TRAIN_START, TRAIN_END),
    ('val',   VAL_START,   VAL_END),
    ('test',  TEST_START,  TEST_END),
]:
    mask = (df_sorted['date'] >= start) & (df_sorted['date'] <= end)
    n = df_sorted.loc[mask, 'entry_optuna'].sum()
    print(f'  {period}: {n} signals')

print()

# Run backtest for each period
optuna_results = []
optuna_equity = {}
optuna_trades = {}

for period, start, end in [
    ('train', TRAIN_START, TRAIN_END),
    ('val',   VAL_START,   VAL_END),
    ('test',  TEST_START,  TEST_END),
]:
    df_p = df_sorted[(df_sorted['date'] >= start) & (df_sorted['date'] <= end)].copy()
    
    if df_p['entry_optuna'].sum() < 3:
        print(f'{period}: no signals, skipping')
        continue
    
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        trades, equity = run_backtest(
            df_p, signal_col='entry_optuna',
            max_hold_days=best_params['max_hold_days'],
        )
    
    optuna_equity[period] = equity
    optuna_trades[period] = trades
    
    if len(trades) >= 1:
        m = calc_metrics(trades, equity, label=period, signal='optuna')
        optuna_results.append(m)
        print(f'{period:8s}: {len(trades):>4} trades | Sharpe={m["sharpe"]:>6.2f} | '
              f'CAGR={m["cagr_%"]:>6.1f}% | MaxDD={m["max_dd_%"]:>6.1f}% | '
              f'HitRate={m["hit_rate_%"]:>5.1f}% | PF={m["profit_factor"]}')
    else:
        print(f'{period:8s}: 0 trades')

optuna_df = pd.DataFrame(optuna_results)


Best parameters: {'vol_consol_days': 3, 'ema_proximity': 0.028401567185994646, 'vol_spike_mult': 1.2048452327966348, 'regime_threshold': 1, 'max_hold_days': 12}

  train: 209 signals
  val: 70 signals
  test: 141 signals



train   :  167 trades | Sharpe=  1.01 | CAGR=  -6.5% | MaxDD= -53.5% | HitRate= 32.3% | PF=0.58
val     :   65 trades | Sharpe=  1.10 | CAGR=   2.5% | MaxDD= -50.4% | HitRate= 40.0% | PF=1.17


test    :  117 trades | Sharpe=  0.79 | CAGR=   9.4% | MaxDD= -44.8% | HitRate= 32.5% | PF=1.33


## 6.7 — Optimization History

In [7]:
# Plot optimization history (matplotlib version — no plotly dependency)
trial_values = [t.value for t in study.trials]
trial_best   = pd.Series(trial_values).cummax().tolist()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: trial-by-trial Sharpe
ax = axes[0]
ax.scatter(range(len(trial_values)), trial_values, s=8, alpha=0.5, color='#2166AC', label='Trial')
ax.plot(range(len(trial_best)), trial_best, color='#D6604D', lw=2, label='Best so far')
ax.axhline(0, color='gray', lw=0.7, linestyle=':')
ax.set_xlabel('Trial')
ax.set_ylabel('Sharpe (train)')
ax.set_title('Optimization History')
ax.legend(fontsize=9)

# Right: parameter distributions (best 20 trials highlighted)
ax2 = axes[1]
best_threshold = np.percentile([t.value for t in study.trials if t.value > -0.5], 80)
best_trials = [t for t in study.trials if t.value >= best_threshold]

params_to_plot = ['ema_proximity', 'vol_spike_mult']
colors = ['#2166AC', '#D6604D']
for (param, color) in zip(params_to_plot, colors):
    vals = [t.params[param] for t in study.trials if t.value > -0.5]
    best_vals = [t.params[param] for t in best_trials]
    ax2.scatter([param] * len(vals), vals, alpha=0.3, s=10, color=color)
    ax2.scatter([param] * len(best_vals), best_vals, alpha=0.8, s=30, color=color, marker='*', label=f'Top-20% {param}')
ax2.set_title('Parameter Spread (top-20% highlighted)')
ax2.set_xlabel('Parameter')
ax2.set_ylabel('Value')

plt.tight_layout()
plt.savefig(FIG_DIR + 'optuna_history.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: optuna_history.png')


Saved: optuna_history.png


## 6.8 — Parameter Importance

In [8]:
# Compute parameter importance from Optuna study
try:
    importances = optuna.importance.get_param_importances(study)
    params_list = list(importances.keys())
    import_vals = list(importances.values())
    
    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.barh(params_list[::-1], import_vals[::-1], color='#2166AC', alpha=0.8)
    ax.set_xlabel('Relative Importance')
    ax.set_title('Parameter Importance (Fanova)')
    for bar, val in zip(bars, import_vals[::-1]):
        ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=9)
    plt.tight_layout()
    plt.savefig(FIG_DIR + 'param_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: param_importance.png')
    print('Importances:', {k: round(v, 3) for k, v in importances.items()})
except Exception as e:
    print(f'Parameter importance failed (need scikit-learn): {e}')
    print('Install with: pip install scikit-learn')


Saved: param_importance.png
Importances: {'vol_spike_mult': np.float64(0.892), 'ema_proximity': np.float64(0.069), 'vol_consol_days': np.float64(0.026), 'regime_threshold': np.float64(0.007), 'max_hold_days': np.float64(0.006)}


## 6.9 — Walk-Forward Validation

In [9]:
# Walk-forward windows: optimize on 2-year train, test on next year
WF_WINDOWS = [
    ('2018-01-01', '2019-12-31', '2020-01-01', '2020-12-31'),
    ('2019-01-01', '2020-12-31', '2021-01-01', '2021-12-31'),
    ('2020-01-01', '2021-12-31', '2022-01-01', '2022-12-31'),
    ('2021-01-01', '2022-12-31', '2023-01-01', '2023-12-31'),
    ('2022-01-01', '2023-12-31', '2024-01-01', '2024-12-31'),
]
WF_TRIALS = 50

def run_walk_forward_window(train_start, train_end, test_start, test_end, n_trials=50):
    """Optimize on [train_start, train_end] and evaluate on [test_start, test_end]."""
    # Build train df and signal index
    wf_train_mask = (
        (df_sorted['date'] >= pd.Timestamp(train_start)) &
        (df_sorted['date'] <= pd.Timestamp(train_end))
    )
    df_wf_train = df_sorted[wf_train_mask].copy().reset_index(drop=True)
    wf_train_idx = np.where(wf_train_mask.values)[0]
    
    # Create study + optimize
    wf_objective = make_objective(df_wf_train, wf_train_idx,
                                  train_start, train_end, n_min_trades=10)
    wf_study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=42),
    )
    wf_study.optimize(wf_objective, n_trials=n_trials, show_progress_bar=False)
    
    wf_best = wf_study.best_trial
    if wf_best.value <= -1.0:
        return None, wf_best.params, wf_best.value
    
    # Evaluate on test period
    wf_signal = build_signal(
        vol_window = wf_best.params['vol_consol_days'],
        ema_prox   = wf_best.params['ema_proximity'],
        spike_mult = wf_best.params['vol_spike_mult'],
        regime_thr = wf_best.params['regime_threshold'],
    )
    
    df_wf_test = df_sorted[
        (df_sorted['date'] >= pd.Timestamp(test_start)) &
        (df_sorted['date'] <= pd.Timestamp(test_end))
    ].copy().reset_index(drop=True)
    
    test_mask_in_sorted = (
        (df_sorted['date'] >= pd.Timestamp(test_start)) &
        (df_sorted['date'] <= pd.Timestamp(test_end))
    ).values
    df_wf_test['entry_wf'] = wf_signal[test_mask_in_sorted]
    
    if df_wf_test['entry_wf'].sum() < 3:
        return None, wf_best.params, wf_best.value
    
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        trades, equity = run_backtest(
            df_wf_test, signal_col='entry_wf',
            max_hold_days=wf_best.params['max_hold_days'],
        )
    
    if len(trades) < 1:
        return None, wf_best.params, wf_best.value
    
    m = calc_metrics(trades, equity, label='oos', signal='wf')
    return m, wf_best.params, wf_best.value

print(f'Walk-forward: {len(WF_WINDOWS)} windows × {WF_TRIALS} trials each')


Walk-forward: 5 windows × 50 trials each


In [10]:
wf_results = []
print(f'{"Window":<30} {"OOS Sharpe":>12} {"OOS CAGR%":>12} {"Trades":>8} {"Train Sharpe":>13}')
print('-' * 80)

t0 = time.time()
for i, (ts, te, vs, ve) in enumerate(WF_WINDOWS):
    metrics, params, train_sharpe = run_walk_forward_window(ts, te, vs, ve, n_trials=WF_TRIALS)
    
    row = {
        'window': f'{ts[:4]}-{te[:4]} → {vs[:4]}-{ve[:4]}',
        'train_sharpe': round(train_sharpe, 3),
        'best_params': params,
    }
    if metrics is not None:
        row.update({
            'oos_sharpe':   metrics.get('sharpe', np.nan),
            'oos_cagr':     metrics.get('cagr_%', np.nan),
            'oos_trades':   metrics.get('n_trades', 0),
            'oos_hit_rate': metrics.get('hit_rate_%', np.nan),
        })
        print(f'{row["window"]:<30} {row["oos_sharpe"]:>12.3f} {row["oos_cagr"]:>12.1f} '
              f'{row["oos_trades"]:>8} {train_sharpe:>13.3f}')
    else:
        row.update({'oos_sharpe': np.nan, 'oos_cagr': np.nan, 'oos_trades': 0, 'oos_hit_rate': np.nan})
        print(f'{row["window"]:<30} {"no trades":>12} {"n/a":>12} {"0":>8} {train_sharpe:>13.3f}')
    
    wf_results.append(row)

elapsed = time.time() - t0
print(f'Walk-forward complete in {elapsed:.1f}s')

wf_df = pd.DataFrame(wf_results)
pos_windows = (wf_df['oos_sharpe'] > 0).sum()
print(f'\n{pos_windows}/{len(WF_WINDOWS)} windows with positive OOS Sharpe')
avg_oos_sharpe = wf_df['oos_sharpe'].mean()
print(f'Average OOS Sharpe: {avg_oos_sharpe:.3f}')


Window                           OOS Sharpe    OOS CAGR%   Trades  Train Sharpe
--------------------------------------------------------------------------------


2018-2019 → 2020-2020                 1.120         19.0       72         0.920


2019-2020 → 2021-2021                 1.100          0.6       59         1.220


2020-2021 → 2022-2022                 0.630         -5.1       24         1.410


2021-2022 → 2023-2023                 1.040         60.1       38         0.860


2022-2023 → 2024-2024                 1.080          6.8       68         0.860
Walk-forward complete in 36.6s

5/5 windows with positive OOS Sharpe
Average OOS Sharpe: 0.994


In [11]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Sharpe by window
ax = axes[0]
windows_short = [r['window'] for r in wf_results]
sharpes = [r['oos_sharpe'] for r in wf_results]
colors = ['#4DAC26' if s > 0 else '#D6604D' for s in sharpes]
bars = ax.bar(range(len(windows_short)), sharpes, color=colors, alpha=0.8, edgecolor='white')
ax.axhline(0, color='black', lw=0.8)
ax.set_xticks(range(len(windows_short)))
ax.set_xticklabels([w.split(' → ')[1] for w in windows_short], rotation=30, ha='right')
ax.set_ylabel('OOS Sharpe Ratio')
ax.set_title(f'Walk-Forward OOS Sharpe ({pos_windows}/{len(WF_WINDOWS)} positive)')
for bar, val in zip(bars, sharpes):
    if not np.isnan(val):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.2f}', ha='center', va='bottom', fontsize=9)

# Right: Train vs OOS Sharpe scatter
ax2 = axes[1]
train_sharpes = [r['train_sharpe'] for r in wf_results]
ax2.scatter(train_sharpes, sharpes, s=80, color='#2166AC', zorder=3)
ax2.axhline(0, color='gray', lw=0.7, linestyle=':')
ax2.axvline(0, color='gray', lw=0.7, linestyle=':')
ax2.set_xlabel('Train Sharpe')
ax2.set_ylabel('OOS Sharpe')
ax2.set_title('Train vs OOS Sharpe (Overfit Scatter)')
for i, (x, y, w) in enumerate(zip(train_sharpes, sharpes, windows_short)):
    if not np.isnan(y):
        ax2.annotate(f'W{i+1}', (x, y), textcoords='offset points', xytext=(5, 5), fontsize=8)

plt.tight_layout()
plt.savefig(FIG_DIR + 'walk_forward_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: walk_forward_results.png')


Saved: walk_forward_results.png


## 6.10 — Final Comparison Table

In [12]:
# Load existing backtest comparison for baseline numbers
try:
    base_df = pd.read_csv(DATA_DIR + 'backtest_comparison.csv')
    # Extract test period rows for key signals
    test_base = base_df[base_df['period'] == 'test'].set_index('signal')
    
    def get_row(sig):
        if sig in test_base.index:
            r = test_base.loc[sig]
            return {
                'config':       sig,
                'trades':       int(r.get('n_trades', 0)),
                'sharpe':       round(float(r.get('sharpe', float('nan'))), 2),
                'cagr_pct':     round(float(r.get('cagr_%', float('nan'))), 1),
                'max_dd_pct':   round(float(r.get('max_dd_%', float('nan'))), 1),
                'hit_rate_pct': round(float(r.get('hit_rate_%', float('nan'))), 1),
                'profit_factor':round(float(r.get('profit_factor', float('nan'))), 2),
            }
        return None

    rows = []
    for sig in ['original', 'relaxed_v2', 'v2_regime>=3', 'v2_regime>=3+SPY']:
        r = get_row(sig)
        if r:
            rows.append(r)
except Exception as e:
    print(f'Could not load backtest_comparison.csv: {e}')
    rows = []

# Add Optuna optimized (test period)
if len(optuna_results) > 0:
    opt_test = next((r for r in optuna_results if r['period'] == 'test'), None)
    if opt_test:
        rows.append({
            'config':       'optuna_optimized',
            'trades':       int(opt_test.get('n_trades', 0)),
            'sharpe':       round(float(opt_test.get('sharpe', float('nan'))), 2),
            'cagr_pct':     round(float(opt_test.get('cagr_%', float('nan'))), 1),
            'max_dd_pct':   round(float(opt_test.get('max_dd_%', float('nan'))), 1),
            'hit_rate_pct': round(float(opt_test.get('hit_rate_%', float('nan'))), 1),
            'profit_factor':round(float(opt_test.get('profit_factor', float('nan'))), 2),
        })

# Add walk-forward average
if len(wf_df) > 0:
    rows.append({
        'config':       'walk_forward_avg_oos',
        'trades':       int(wf_df['oos_trades'].mean()),
        'sharpe':       round(float(wf_df['oos_sharpe'].mean()), 2),
        'cagr_pct':     round(float(wf_df['oos_cagr'].mean()), 1),
        'max_dd_pct':   float('nan'),
        'hit_rate_pct': round(float(wf_df['oos_hit_rate'].mean()), 1),
        'profit_factor':float('nan'),
    })

final_df = pd.DataFrame(rows)
print('FINAL COMPARISON TABLE — Test Period (2022-2024) / Walk-Forward OOS')
print('='*90)
print(final_df.to_string(index=False))

# Save
final_df.to_csv(DATA_DIR + 'final_comparison.csv', index=False)
print(f'\nSaved: final_comparison.csv')


FINAL COMPARISON TABLE — Test Period (2022-2024) / Walk-Forward OOS
              config  trades  sharpe  cagr_pct  max_dd_pct  hit_rate_pct  profit_factor
            original       4    0.20       1.3       -19.2          75.0          50.04
          relaxed_v2     164    0.98       2.4       -55.6          40.2           0.76
        v2_regime>=3     151    1.00       1.0       -58.1          36.4           0.66
    v2_regime>=3+SPY     151    1.00       0.6       -58.1          36.4           0.66
    optuna_optimized     117    0.79       9.4       -44.8          32.5           1.33
walk_forward_avg_oos      52    0.99      16.3         NaN          40.0            NaN

Saved: final_comparison.csv


In [13]:
# Bar chart of Sharpe ratios
plot_rows = final_df[final_df['config'] != 'walk_forward_avg_oos'].copy()
wf_row = final_df[final_df['config'] == 'walk_forward_avg_oos']

fig, ax = plt.subplots(figsize=(12, 5))
bar_colors = ['#AAAAAA', '#4DAC26', '#2166AC', '#5AAE61', '#D6604D', '#762A83']
bars = ax.bar(
    range(len(plot_rows)),
    plot_rows['sharpe'].values,
    color=bar_colors[:len(plot_rows)],
    alpha=0.85,
    edgecolor='white',
    width=0.6,
)
if len(wf_row) > 0:
    ax.axhline(float(wf_row['sharpe'].iloc[0]), color='#B2182B', lw=2,
               linestyle='--', label=f'Walk-Fwd Avg OOS: {float(wf_row["sharpe"].iloc[0]):.2f}')
ax.axhline(0, color='black', lw=0.7)
ax.set_xticks(range(len(plot_rows)))
ax.set_xticklabels(plot_rows['config'].tolist(), rotation=25, ha='right', fontsize=9)
ax.set_ylabel('Sharpe Ratio (Test Period 2022-2024)')
ax.set_title('DriftFire — Strategy Evolution (Test Period Sharpe)')
ax.legend(fontsize=9)

for bar, (_, row) in zip(bars, plot_rows.iterrows()):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.01,
            f'{h:.2f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(FIG_DIR + 'final_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: final_comparison.png')


Saved: final_comparison.png


## 6.11 — Save All Results

In [14]:
# Save walk-forward results
wf_save = wf_df.drop(columns=['best_params'], errors='ignore')
wf_save.to_csv(DATA_DIR + 'walk_forward_results.csv', index=False)
print(f'Saved: walk_forward_results.csv')

# Save best optuna params
import json as _json
best_params_out = {
    'vol_consol_days': int(best_params['vol_consol_days']),
    'ema_proximity':   float(best_params['ema_proximity']),
    'vol_spike_mult':  float(best_params['vol_spike_mult']),
    'regime_threshold':int(best_params['regime_threshold']),
    'max_hold_days':   int(best_params['max_hold_days']),
    'train_sharpe':    float(study.best_value),
}
with open(DATA_DIR + 'optuna_best_params.json', 'w') as f:
    _json.dump(best_params_out, f, indent=2)
print(f'Saved: optuna_best_params.json')
print()
print('=' * 60)
print('SUMMARY')
print('=' * 60)
print(f'Main optimization: Sharpe={study.best_value:.3f} (train)')
print(f'Best params: {best_params_out}')
print()
print(f'Walk-forward: {pos_windows}/{len(WF_WINDOWS)} windows positive OOS')
print(f'  Avg OOS Sharpe: {avg_oos_sharpe:.3f}')
print()
print('Final comparison:')
print(final_df[['config','trades','sharpe','cagr_pct']].to_string(index=False))


Saved: walk_forward_results.csv
Saved: optuna_best_params.json

SUMMARY
Main optimization: Sharpe=1.010 (train)
Best params: {'vol_consol_days': 3, 'ema_proximity': 0.028401567185994646, 'vol_spike_mult': 1.2048452327966348, 'regime_threshold': 1, 'max_hold_days': 12, 'train_sharpe': 1.01}

Walk-forward: 5/5 windows positive OOS
  Avg OOS Sharpe: 0.994

Final comparison:
              config  trades  sharpe  cagr_pct
            original       4    0.20       1.3
          relaxed_v2     164    0.98       2.4
        v2_regime>=3     151    1.00       1.0
    v2_regime>=3+SPY     151    1.00       0.6
    optuna_optimized     117    0.79       9.4
walk_forward_avg_oos      52    0.99      16.3
